# Tutorial for Training/Deploying the Physical CartPole Demo

This notebook follows the demo-hls4ml-25 README (Steps 1–7)

> Use the navigation links below to jump around quickly.


## Navigation
- [Step 0 — Notebook prerequisites](#step-0-notebook-prerequisites)
- [Step 1 — Environment Setup](#step-1-environment-setup)
- [Step 2 — Training Neural Network Controller](#step-2-training-neural-network-controller)
- [Step 3 — Running the Cartpole Simulator](#step-3-running-the-cartpole-simulator)
- [Step 4 — Convert Neural Network Controller with hls4ml](#step-4-convert-neural-network-controller-with-hls4ml)
- [Step 5 — Testing Model on PC (Local Hardware)](#step-5-testing-model-on-pc-local-hardware)
- [Step 6 — Implementation (Vivado/Vitis)](#step-6-implementation-vivadovitis)
- [Step 7 — Load Image on SD card and onto FPGA](#step-7)


## Step 0: Notebook prerequisites 

- Please make sure you have Conda installed
    - Refer to [this article](https://docs.conda.io/projects/conda/en/latest/user-guide/install/index.html) for instructions on setting it up
- Ensure you have access to these programs
    - Vivado 2020.1 
    - Vitis 2020.1


## Step 1: Environment Setup

### Conda environment
In your terminal run these
```bash
conda create -n physical_cartpole python=3.10
conda activate physical_cartpole
```
### Install Packages
These are just the necessary packages for training, we will install the GUI/Simulation packages later
<!-- GUI extras often needed in notebook environments:
## %pip install watchdog pydot graphviz PyQt6 -->
**Lab server note:** if your environment is pre-configured, you can skip this step.


In [4]:
# Install base dependencies
%pip install -r "../Driver/CartPoleSimulation/CPS_list_of_packages.txt"

# Notebook/GUI extras used in this walkthrough
%pip install seaborn watchdog pydot graphviz PyQt6


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### 1.1 Choose experiment + model name & set paths for later

Edit these values directly in the next code cell:
- `EXPERIMENT_NAME = "Experiment-1"`
- `NET_NAME = "Dense-7IN-32H1-32H2-1OUT-0"`

If you want a different experiment/model, change those strings and rerun the setup cell.

Why paths are resolved here:
- Jupyter can start with different current working directories (repo root, a subfolder, or an external launch directory).
- Downstream steps call scripts with relative paths (`step3.sh`, SI_Toolkit configs, Vivado/Vitis scripts).
- We resolve `REPO` once in this setup cell and all later cells reuse it.


In [5]:
import os, shutil
import sys

from pathlib import Path
from typing import Optional

def find_repo_root(start: Optional[Path] = None) -> Path:
    """Locate the physical-cartpole repo from varied notebook launch directories."""
    cursor = (start or Path.cwd()).resolve()
    for candidate in [cursor, *cursor.parents]:
        if (candidate / "Driver" / "CartPoleSimulation").exists() and (candidate / "README-demo-hls4ml-25.md").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate physical-cartpole repo root. "
        "Start Jupyter from this repository or one of its subdirectories."
    )

# Resolve once so all later cells use the same root path.
# Supported contexts:
# 1) Notebook launched from repo root
# 2) Notebook launched from a subdirectory in this repo
# 3) Jupyter launched elsewhere, then notebook opened from this repo
REPO = find_repo_root()

SCRIPTS = REPO / "scripts"

if str(SCRIPTS) not in sys.path:

    sys.path.insert(0, str(SCRIPTS))
SIM = REPO / "Driver" / "CartPoleSimulation"
SI_ASF = SIM / "SI_Toolkit_ASF"
WORKSPACE = SI_ASF / "Experiments"
HLS_CONFIG = SI_ASF / "config_hls.yml"

# Set the experiment name here (edit this string for your run)
EXPERIMENT_NAME = "Experiment-1"

# Set the neural-network model name here (edit this string for your run)
NET_NAME = "Dense-7IN-32H1-32H2-1OUT-0"

# Path to seed (template) experiment that is provided
SEED_EXPERIMENT = REPO / "Experiment-1"

# Path to active experiment workspace used by SI_Toolkit
ACTIVE_EXPERIMENT = WORKSPACE / EXPERIMENT_NAME
MODELS_DIR = ACTIVE_EXPERIMENT / "Models"

# Ensure repo-local SI_Toolkit is importable without depending on editable installs.
si_src = SIM / "SI_Toolkit" / "src"
if str(si_src) not in sys.path:
    sys.path.insert(0, str(si_src))

# Ensure SI_Toolkit_ASF imports resolve from this repository checkout.
if str(SIM) not in sys.path:
    sys.path.insert(0, str(SIM))

# Echo resolved paths and selected model for verification
print("REPO:", REPO)
print("SIM:", SIM)
print("SI_ASF:", SI_ASF)
print("ACTIVE_EXPERIMENT:", ACTIVE_EXPERIMENT)
print("MODELS_DIR:", MODELS_DIR)
print("HLS_CONFIG:", HLS_CONFIG)
print("NET_NAME:", NET_NAME)

REPO: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole
SIM: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation
SI_ASF: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF
ACTIVE_EXPERIMENT: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/Experiments/Experiment-1
MODELS_DIR: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/Experiments/Experiment-1/Models
HLS_CONFIG: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/config_hls.yml
NET_NAME: Dense-7IN-32H1-32H2-1OUT-0


### 1.2 Move/copy seed experiment into the SI_Toolkit experiments folder

This shell scripts effectively ensure the experiment lives under `SI_Toolkit_ASF/Experiments/`.
We do a safe copy **only if missing**, to avoid overwriting trained artifacts.


In [6]:
import shutil

if not ACTIVE_EXPERIMENT.exists():
    shutil.copytree(SEED_EXPERIMENT, ACTIVE_EXPERIMENT)
    print("Copied ./Experiment-1 →", ACTIVE_EXPERIMENT)
else:
    print("Active experiment already exists: not overwriting")

Active experiment already exists: not overwriting


## Step 2: Training Neural Network Controller

### Dataset location
The Seed dataset is initially located at: `./Experiment-1` in the root directory which contains:
- recorded trajectories (CSV)
- a known good model configuration

Now there are **two paths**:

### Path A: Use precomputed model
- Use a pre-trained model from the seed folder `../Experiment-1/Models` (chosen in step 1)
- [Skip to the simulation (Step 3)](#step-3-running-the-cartpole-simulator)

### Path B: Train the neural network
- Follow the next steps

### 2.1 Training Configuration (config_training.yml)

Run this cell to see the current training configurations in:
`Driver/CartPoleSimulation/SI_Toolkit_ASF/config_training.yml`

Please refer to the [training_configurations_walkthrough](training_configurations_walkthrough.ipynb) notebook for more details on the training configurations

In [ ]:
import yaml
from pprint import pprint

training_config_path = SI_ASF / "config_training.yml"

with training_config_path.open("r") as f:
    cfg = yaml.safe_load(f)

print("Loaded:", training_config_path)
pprint(cfg)

Below is an **optional** cell that updates the yaml's experiment path if you changed it

In [ ]:
from demo_helpers import set_training_experiment_path

# Updates config_training.yml experiment path key using existing key-search behavior.
set_training_experiment_path(
    cfg_path=SI_ASF / "config_training.yml",
    active_experiment=ACTIVE_EXPERIMENT,
)


### 2.2 Data normalization

Neural networks train much more reliably when each feature is on a comparable numeric scale.

In this project, **normalization statistics are computed from the `Train/` CSVs** inside the active experiment folder.  
Those statistics are saved to `NormalizationInfo/` and then reused consistently for:

- training 
- simulation inference
- FPGA/HLS deployment

The function that does this is `SI_Toolkit.load_and_normalize.calculate_normalization_info(...)`.


In [ ]:
# Load the training configuration
import yaml

cfg_path = SI_ASF / "config_training.yml"
cfg = yaml.safe_load(cfg_path.read_text())

# The two path keys
print("PATH_TO_EXPERIMENT_FOLDERS =", cfg["paths"]["PATH_TO_EXPERIMENT_FOLDERS"])
print("path_to_experiment         =", cfg["paths"]["path_to_experiment"])
print("DATA_FOLDER                =", cfg["paths"]["DATA_FOLDER"])

train_dir = Path(cfg["paths"]["PATH_TO_EXPERIMENT_FOLDERS"]) / cfg["paths"]["path_to_experiment"] / cfg["paths"]["DATA_FOLDER"] / "Train"
print("\nTrain directory (resolved):", train_dir.resolve())


In [ ]:
# Identify the CSVs that are used for normalization
import glob

train_csvs = sorted(glob.glob(str(train_dir / "*.csv")))
print(f"Found {len(train_csvs)} training CSV files.")
for p in train_csvs[:10]:
    print(" -", p)
if len(train_csvs) > 10:
    print(f" ... ({len(train_csvs)-10} more)")


In [ ]:
from SI_Toolkit.load_and_normalize import get_paths_to_datafiles, load_data

# Build list of CSV paths
train_paths = get_paths_to_datafiles(str(train_dir))
val_dir = train_dir.parent / "Validation"
test_dir = train_dir.parent / "Test"

# Load CSVs into DataFrames
training_dfs = load_data(train_paths)

validation_dfs = load_data(get_paths_to_datafiles(str(val_dir)))
test_dfs       = load_data(get_paths_to_datafiles(str(test_dir)))

print("Loaded training files:", len(training_dfs))
print("Example columns:", training_dfs[0].columns.tolist())

#### 2.2.1 Compute normalization statistics + write `NormalizationInfo/NI_*.csv`

By default the function:
- concatenates all Train CSVs into one DataFrame
- drops a `time` column (if present)
- computes per-feature **mean/std/min/max**
- optionally applies a **user correction hook** from `SI_Toolkit_ASF/ToolkitCustomization/...`
- writes a timestamped `NI_YYYY-MM-DD_HH-MM-SS.csv`
- optionally saves histogram PNGs per feature


In [ ]:
# Run the normalization (preprocessing) step
from SI_Toolkit.load_and_normalize import calculate_normalization_info

# Keep histograms ON for now, optionally turn OFF for speed in automated runs
df_norm_info, norm_csv_path = calculate_normalization_info(
    config=cfg,
    plot_histograms=True,
    user_correction=False, 
)

print("Wrote normalization CSV:", norm_csv_path)
display(df_norm_info)

#### 2.2.2 How to read `df_norm_info`

`df_norm_info` is indexed by the statistic (`mean`, `std`, `min`, `max`).  
Each column is a feature from your training CSVs (excluding `time`).

Typical z-score normalization uses:

- **normalize**:  \(x_{norm} = (x - \mu) / \sigma\)
- **denormalize**: \(x = x_{norm} \cdot \sigma + \mu\)

The exact columns used as inputs/targets are determined by your training config and the model wrapper,
but the statistics come from the raw CSV columns.


#### 2.2.3 Where the histograms went

If `plot_histograms=True`, the function saves one histogram per feature to:

`.../NormalizationInfo/histograms/<feature>.png`

In [ ]:
# Show a couple histogram images
from pathlib import Path

hist_dir = Path(norm_csv_path).parent / "histograms"
print("Histogram dir:", hist_dir)

if hist_dir.exists():
    pngs = sorted(hist_dir.glob("*.png"))
    print("Found", len(pngs), "histograms.")
else:
    print("No histograms folder found (plot_histograms may be False).")


### 2.3 Train the neural network
At this point, we have written a normalization file from the **Train** split. Now we go through the training flow.

Training is config driven, uses the normalization statistics computed above, and writes a fully reproducible model folder under `Models/`.

#### 2.3.1 Load training arguments (YAML + CLI overrides)

Training is config driven: `args()` loads `config_training.yml` defaults and applies any CLI overrides.
We print the resolved paths and split files so you can confirm the **active experiment**.


In [ ]:
## Set the correct path to run training 
import os
from pathlib import Path

if "SIM" not in globals():
    raise RuntimeError("Run Step 1.1 first so shared paths (REPO/SIM) are defined.")

print("Setting working directory to:", SIM)
os.chdir(SIM)

print("Now cwd =", Path.cwd())
print("Config exists? ", (SIM / "SI_Toolkit_ASF" / "config_training.yml").exists())


In [ ]:
from SI_Toolkit.Functions.General.load_parameters_for_training import args 
from SI_Toolkit.Functions.General.Initialization import set_seed

# Save Jupyter argv 
_argv_backup = sys.argv.copy()

# Make argparse think we're running with no CLI args
sys.argv = [sys.argv[0]]

# This parses the yaml file for our training details. 
# here we can change things like batch size and epochs
a = args()

# Sets random seed
set_seed(a)

# Restore argv
sys.argv = _argv_backup
print("Key training settings:")
for k in ["net_name", "library", "path_to_models", "training_files", "validation_files", "test_files", "config_path"]:
    if hasattr(a, k):
        print(f"  {k}: {getattr(a, k)}")


#### 2.3.2 Instantiate the model and inspect data

`get_net(a)` builds the network and a `net_info` object that describes naming, paths, and whether normalization is enabled. `create_full_name(...)` generates the run folder name.


In [ ]:
from SI_Toolkit.Functions.General.Initialization import get_net, create_full_name

net, net_info = get_net(a)
create_full_name(net_info, a.path_to_models)

print("Model full name:", net_info.net_full_name)
print("Backend library:", net_info.library)
print("Will normalize:", getattr(net_info, "normalize", None))
print("Models root folder:", a.path_to_models)
print("This run will be saved under:", f"{a.path_to_models}/{net_info.net_full_name}")


#### 2.3.3 Load normalization info

This is why normalization must run first: training needs the mean/std (and related vectors) to scale features.
These vectors are also reused later for consistent inference (including HLS/firmware).


In [ ]:
from SI_Toolkit.Functions.General.Initialization import get_norm_info_for_net
from SI_Toolkit.Functions.General.Normalising import write_out_normalization_vectors

normalization_info = get_norm_info_for_net(net_info, files_for_normalization=a.training_files)
write_out_normalization_vectors(normalization_info, net_info)

print("Normalization info loaded and vectors written for:", net_info.net_full_name)

#### 2.3.4 Load Train/Validate/Test CSVs 

Here we resolve file lists for each split, load them as DataFrames, and print basic counts. This connects CSV logs to the training tensors.


In [ ]:
from SI_Toolkit.load_and_normalize import load_data, get_paths_to_datafiles
from SI_Toolkit.Functions.General.Initialization import create_log_file
import os

train_paths = get_paths_to_datafiles(a.training_files)
val_paths   = get_paths_to_datafiles(a.validation_files)
test_paths  = get_paths_to_datafiles(a.test_files)

training_dfs   = load_data(train_paths)
validation_dfs = load_data(val_paths)
test_dfs       = load_data(test_paths)

run_dir = os.path.join(a.path_to_models, net_info.net_full_name)
os.makedirs(run_dir, exist_ok=True)

create_log_file(net_info, a, training_dfs)

def nrows(dfs):
    return sum(len(df) for df in dfs) if isinstance(dfs, list) else len(dfs)

print("Train files:", len(train_paths), "rows:", nrows(training_dfs))
print("Val files:  ", len(val_paths),   "rows:", nrows(validation_dfs))
print("Test files: ", len(test_paths),  "rows:", nrows(test_dfs))

example_cols = training_dfs[0].columns if isinstance(training_dfs, list) else training_dfs.columns
print("Example columns:", list(example_cols))


#### 2.3.5 Add Previous Control Input (`Q_applied_-1`) and Repair Normalization

This step augments each dataset with a one timestep delayed control input.
For every DataFrame, we sort by time, shift `Q_applied` by one step to create `Q_applied_-1`, and drop the first row (which has no previous value).

Because `Q_applied_-1` is derived after loading CSVs, older normalization files may have missing/NaN stats for this column.
The next code cell automatically repairs normalization statistics for any required input/output column that is missing or nonfinite.


In [ ]:
import numpy as np
import pandas as pd

def add_prev_Q_applied(dfs, time_col="time"):
    for df in dfs:
        if "Q_applied_-1" in df.columns:
            continue

        # Ensure sorted by time just in case
        if time_col in df.columns:
            df.sort_values(time_col, inplace=True)

        df["Q_applied_-1"] = df["Q_applied"].shift(1)

        # First row has no previous value -> drop it
        df.dropna(subset=["Q_applied_-1"], inplace=True)
        df.reset_index(drop=True, inplace=True)

def ensure_normalization_stats(normalization_info, dfs_for_stats, required_columns):
    repaired = []

    for col in required_columns:
        missing_col = col not in normalization_info.columns
        invalid_col = (not missing_col) and normalization_info[col].isna().any()

        if not (missing_col or invalid_col):
            continue

        values = []
        for df in dfs_for_stats:
            if col in df.columns:
                s = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64, copy=False)
                s = s[np.isfinite(s)]
                if s.size > 0:
                    values.append(s)

        if not values:
            raise ValueError(
                f"Cannot repair normalization for '{col}': column missing/non-finite in all provided DataFrames."
            )

        x = np.concatenate(values)
        normalization_info.loc["mean", col] = float(np.mean(x))
        normalization_info.loc["std", col] = float(np.std(x))
        normalization_info.loc["max", col] = float(np.max(x))
        normalization_info.loc["min", col] = float(np.min(x))
        repaired.append(col)

    return repaired

add_prev_Q_applied(training_dfs)
add_prev_Q_applied(validation_dfs)
add_prev_Q_applied(test_dfs)

required_cols = sorted(set(net_info.inputs) | set(net_info.outputs))
repaired_cols = ensure_normalization_stats(
    normalization_info,
    training_dfs + validation_dfs,
    required_cols,
)

if repaired_cols:
    print("Repaired normalization stats for:", repaired_cols)
else:
    print("Normalization stats already valid for all required columns")

# Verify one training dataframe
df0 = training_dfs[0]
print(df0[["time", "Q_applied", "Q_applied_-1", "Q_calculated"]].head(5))

#### 2.3.6 Run the training loop

This is the real training loop used by `train_network()`: it calls `Training.train_network_core(...)`.
It returns loss curves used to generate the training plot.

A preflight data check runs in the previous cell and blocks training if non-finite values are detected.

**Note:** Training can take a long time.


In [ ]:
if net_info.library == "TF":
    import SI_Toolkit.Functions.TF.Training as Training
else:
    import SI_Toolkit.Functions.Pytorch.Training as Training

from SI_Toolkit.Functions.General.TerminalContentManager import TerminalContentManager

with TerminalContentManager(os.path.join(run_dir, "terminal_output.txt")):
    loss, val_loss, post_epoch_loss = Training.train_network_core(
        net, net_info,
        training_dfs,
        validation_dfs,
        test_dfs,
        normalization_info,
        a
    )

print("Final validation loss:", val_loss[-1])

#### 2.5.6 Plot losses and point to saved artifacts

After training finishes, the run folder under `Models/` contains:
- the trained model/checkpoints
- `terminal_output.txt` (captured outputs)
- copied config + training script
- `training_curve.png`


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

plt.figure()
plt.plot(loss, label="train")
plt.plot(val_loss, label="val")
plt.yscale("log")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title(net_info.net_full_name)
plt.savefig(os.path.join(net_info.path_to_net, "training_curve.png"))
plt.show()

model_dir = Path(a.path_to_models) / net_info.net_full_name
print("Model artifacts saved in:", model_dir)
print("Look for: training_curve.png, terminal_output.txt, config copy, checkpoints/models")

NET_NAME = net_info.net_full_name
print("NET_NAME updated to trained run:", NET_NAME)


## Step 3: Running the Cartpole Simulator
In this section, we will primarily focus on using the Cartpole Simulator to test the performance of trained neural network controllers. This simulator not only allows for performance evaluation but can also be used to generate new datasets for further training. Here, we will describe how to effectively use the simulator to assess your model's capabilities.

### 3.1 Choose the Model
   - Run the program with the desired model name as an argument to select a specific trained model. If no model name is provided, the default pre-trained model will be used.
   - Available model names can be found in the following folder:
     ```
     Driver/CartPoleSimulation/SI_Toolkit_ASF/Experiments/Experiment-1/Models
     ```

### 3.2 Run the GUI
- Make sure you have access to a display variable


In [ ]:
from demo_helpers import ensure_notebook_repo_context, run_step3_launcher

# Ensures Step 1.1 repo context exists and repo imports resolve from this checkout.
REPO_PATH = ensure_notebook_repo_context(
    globals(),
    missing_repo_message="Run Step 1.1 first so REPO is defined.",
    ensure_sys_path=False,
)

# Runs step3.sh from repo root with optional NET_NAME arg and unchanged cwd semantics.
run_step3_launcher(repo=REPO_PATH, net_name=NET_NAME)


## Step 4: Convert Neural Network Controller with hls4ml

This section converts the selected/trained neural controller into HLS using the SI_Toolkit + hls4ml pipeline.


### 4.1 Select the model to convert

If Step 2 training was run in this notebook, we use that exact run name (`net_info.net_full_name`).
Otherwise we fall back to `NET_NAME`.


In [7]:
import os
from demo_helpers import resolve_selected_model_name

# Chooses trained model name when Step 2 Path B ran; otherwise keeps NET_NAME from Step 1.
SELECTED_NET_NAME = resolve_selected_model_name(globals(), NET_NAME)

# Single source of truth for notebook-wide Xilinx tool version.
XILINX_TOOL_VERSION = "2020.1"

# Fixes bug with DEBUG enviroment var getting set to 'release'
os.environ.pop("DEBUG", None)

# Prefer Experiment-1 models dir if present, otherwise keep active experiment path
if (SI_ASF / "Experiments" / "Experiment-1" / "Models").exists():
    HLS_MODELS_DIR = SI_ASF / "Experiments" / "Experiment-1" / "Models"
else:
    HLS_MODELS_DIR = MODELS_DIR

print("Xilinx tool version target:", XILINX_TOOL_VERSION)
print("SELECTED_NET_NAME:", SELECTED_NET_NAME)
print("HLS_MODELS_DIR:", HLS_MODELS_DIR)
print("Model folder exists:", (HLS_MODELS_DIR / SELECTED_NET_NAME).exists())


Xilinx tool version target: 2020.1
SELECTED_NET_NAME: Dense-7IN-32H1-32H2-1OUT-0
HLS_MODELS_DIR: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/Experiments/Experiment-1/Models
Model folder exists: True


### 4.2 `config_hls.yml` in detail

Use the [hls4ml_config_hls_walkthrough](hls4ml_config_hls_walkthrough.ipynb) notebook to learn more about these parameters.

That notebook explains each key and how it impacts synthesis/resource/latency behavior.

#### 4.2.1 Load and inspect current `config_hls.yml`


In [8]:
import yaml

with HLS_CONFIG.open("r") as f:
    hls_cfg = yaml.safe_load(f)

print("Loaded:", HLS_CONFIG)
for k in [
    "path_to_hls_installation",
    "path_to_models",
    "net_name",
    "batch_size",
    "Strategy",
    "ReuseFactor",
    "backend",
    "output_dir",
]:
    print(f"{k}: {hls_cfg.get(k)}")

print("PRECISION:")
for k, v in hls_cfg.get("PRECISION", {}).items():
    print(f"  {k}: {v}")


Loaded: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/config_hls.yml
path_to_hls_installation: /tools/Xilinx/Vivado/2020.1/bin
path_to_models: SI_Toolkit_ASF/Experiments/Experiment-1/Models
net_name: Dense-7IN-32H1-32H2-1OUT-0
batch_size: 1
Strategy: Resources
ReuseFactor: 32
backend: Vivado
output_dir: ../../HLS4ML/Dense-7IN-32H1-32H2-1OUT-0_20260302_175241
PRECISION:
  input_and_output: ap_fixed<12,2>
  activations: ap_fixed<12,1>
  weights_and_biases: ap_fixed<14,4>
  intermediate_results: ap_fixed<18,6>


#### 4.2.2 How each configuration family affects synthesis

- `PRECISION`: There are fixed point width/integer choices, larger widths usually improve accuracy, but increase resource usage.
- `Strategy`: `Resources` tends to trade latency for lower area; `Latency` pushes more parallelism.
- `ReuseFactor`: Higher reuse typically lowers DSP usage and increases latency.
- `backend`/`part`: Select synthesis backend and target FPGA part.
- `path_to_models` + `net_name`: Choose which trained model checkpoint folder is converted.
- `output_dir`: Controls where generated project and reports are emitted.


#### 4.2.3 Resolve run specific paths (paths, model, output folder)


In [9]:
from demo_helpers import prepare_hls_toolchain_context

# Resolves Vivado/GCC paths and HLS output locations with the existing fallback order.
STEP4_TOOLCHAIN = prepare_hls_toolchain_context(
    repo=REPO,
    sim=SIM,
    hls_cfg=hls_cfg,
    hls_models_dir=HLS_MODELS_DIR,
    selected_net_name=SELECTED_NET_NAME,
    xilinx_tool_version="2020.1",
)

RUN_TAG = STEP4_TOOLCHAIN["run_tag"]
XILINX_TOOL_VERSION = STEP4_TOOLCHAIN["xilinx_tool_version"]
vivado_bin = STEP4_TOOLCHAIN["vivado_bin"]
vivado_version_line = STEP4_TOOLCHAIN["vivado_version_line"]
HLS_OUTPUT_NAME = STEP4_TOOLCHAIN["hls_output_name"]
HLS_OUTPUT_ABS = STEP4_TOOLCHAIN["hls_output_abs"]
HLS_OUTPUT_REL = STEP4_TOOLCHAIN["hls_output_rel"]


Xilinx tool version target: 2020.1
Resolved Vivado bin: /tools/Xilinx/Vivado/2020.1/bin
Resolved Vivado version: Vivado v2020.1 (64-bit)
Resolved AP_GCC_PATH: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/.cache/vivado_gcc_wrapper_2020_1
Resolved real GCC bin: /tools/Xilinx/Vivado/2020.1/tps/lnx64/gcc-6.2.0/bin
Resolved model dir: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/Experiments/Experiment-1/Models/Dense-7IN-32H1-32H2-1OUT-0
Resolved output dir: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/HLS4ML/Dense-7IN-32H1-32H2-1OUT-0_20260305_182640
Resolved output rel: ../../HLS4ML/Dense-7IN-32H1-32H2-1OUT-0_20260305_182640


#### 4.2.4 Backup and write `config_hls.yml`


In [10]:
from demo_helpers import update_hls_config_with_backup

# Backs up and rewrites config_hls.yml using the same keys and relative output path behavior.
backup_path = update_hls_config_with_backup(
    hls_config=HLS_CONFIG,
    hls_cfg=hls_cfg,
    vivado_bin=vivado_bin,
    hls_models_dir=HLS_MODELS_DIR,
    sim=SIM,
    selected_net_name=SELECTED_NET_NAME,
    hls_output_rel=HLS_OUTPUT_REL,
    run_tag=RUN_TAG,
)


Backup created: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/config_hls.yml.bak_20260305_182640
Updated config: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/config_hls.yml


#### 4.3 Go through the conversion pipeline

1. load `config_hls.yml`
2. load model via `get_net(...)`
3. copy model artifacts to `output_dir`
4. run `convert_model_with_hls4ml(...)`
5. `compile`/`build`/Vivado report through hls4ml

Note: Its reccomended to browse the outputs of this cell for details like resource utilization and preformance estimate


In [11]:
import hls4ml
import os
import shutil
import yaml
from pathlib import Path
from types import SimpleNamespace

from SI_Toolkit.Functions.General.Initialization import get_net
from SI_Toolkit.Functions.General.TerminalContentManager import TerminalContentManager

os.environ["OSTYPE"] = "linux-gnu"

cfg_path = SIM / "SI_Toolkit_ASF" / "config_hls.yml"
with cfg_path.open("r") as f:
    cfg_local = yaml.safe_load(f)

a_local = SimpleNamespace()
a_local.path_to_models = cfg_local["path_to_models"]
a_local.net_name = cfg_local["net_name"]
batch_size = cfg_local["batch_size"]

print("Config path:", cfg_path)
print("Using model:", a_local.net_name)
print("Using model root:", a_local.path_to_models)
print("Output dir:", cfg_local["output_dir"])
print("AP_GCC_PATH:", os.environ.get("AP_GCC_PATH", "<unset>"))
if os.environ.get("CARTPOLE_REAL_AP_GCC_PATH"):
    print("Real GCC bin:", os.environ["CARTPOLE_REAL_AP_GCC_PATH"])

old_cwd = Path.cwd()
os.chdir(SIM)
try:
    from SI_Toolkit.HLS4ML.hls4ml_functions import convert_model_with_hls4ml

    model_dir = Path(a_local.path_to_models) / a_local.net_name
    if not model_dir.exists():
        raise FileNotFoundError(f"Model directory does not exist: {model_dir}")

    ckpt_candidates = [
        model_dir / f"{a_local.net_name}.ckpt",
        model_dir / f"{a_local.net_name}.ckpt.index",
        model_dir / "ckpt.ckpt",
        model_dir / "ckpt.ckpt.index",
        model_dir / f"{a_local.net_name}.pt",
        model_dir / "ckpt.pt",
    ]
    with TerminalContentManager(os.path.join(cfg_local["output_dir"], "terminal_output.txt")):
        net_local, net_info_local = get_net(
            a_local,
            time_series_length=1,
            batch_size=batch_size,
            stateful=True,
            remove_redundant_dimensions=True,
        )

        path_to_network = os.path.join(a_local.path_to_models, a_local.net_name)
        path_to_hls_network = os.path.join(cfg_local["output_dir"], a_local.net_name)
        shutil.copytree(path_to_network, path_to_hls_network, dirs_exist_ok=True)

        hls_model_local, hls_model_cfg = convert_model_with_hls4ml(net_local)
        hls4ml.utils.plot_model(hls_model_local, show_shapes=True, show_precision=True, to_file=None)

        hls_model_local.build(
            reset      = True,  # Reuse existing HLS project directory if it already exists.
            csim       = True,  # Run C simulation (functional check before RTL generation).
            synth      = True,  # Run HLS synthesis (C/C++ -> RTL with resource/latency estimates).
            cosim      = True,  # Run C/RTL co-simulation to compare C model vs generated RTL.
            validation = True,  # Run hls4ml validation checks on generated outputs.
            export     = True,  # Package/export generated IP/project artifacts to be used with Vivado.
            vsynth     = True,  # Run downstream Vivado synthesis on exported design.
        )
        hls4ml.report.read_vivado_report(cfg_local["output_dir"])
finally:
    os.chdir(old_cwd)

print("Conversion completed")


Config path: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/config_hls.yml
Using model: Dense-7IN-32H1-32H2-1OUT-0
Using model root: SI_Toolkit_ASF/Experiments/Experiment-1/Models
Output dir: ../../HLS4ML/Dense-7IN-32H1-32H2-1OUT-0_20260305_182640
AP_GCC_PATH: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/.cache/vivado_gcc_wrapper_2020_1
Real GCC bin: /tools/Xilinx/Vivado/2020.1/tps/lnx64/gcc-6.2.0/bin
Loading a pretrained network with the full name  Dense-7IN-32H1-32H2-1OUT-0  from  SI_Toolkit_ASF/Experiments/Experiment-1/Models

Inputs to the loaded network: angleD, angle_cos, angle_sin, position, positionD, target_equilibrium, target_position
Outputs from the loaded network: Q_calculated



2026-03-05 18:26:50.230775: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-05 18:26:50.234711: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-05 18:26:50.276478: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-05 18:26:50.276512: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-05 18:26:50.276543: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

Constructed a standard neural network of type Dense, with 2 hidden layers with sizes 32, 32 respectively.
Loading Model:  SI_Toolkit_ASF/Experiments/Experiment-1/Models/Dense-7IN-32H1-32H2-1OUT-0/ckpt.ckpt

Model loaded from a checkpoint.
Model
  Precision:         ap_fixed<18,6>
  ReuseFactor:       32
  Strategy:          Resources
  BramFactor:        1000000000
  TraceOutput:       False
LayerName
  input_1
    Trace:           False
    Precision
      result:        ap_fixed<12,2>
      weight:        ap_fixed<14,4>
      bias:          ap_fixed<14,4>
  layers_0
    Trace:           False
    Precision
      result:        ap_fixed<18,6>
      weight:        ap_fixed<14,4>
      bias:          ap_fixed<14,4>
  layers_0_linear
    Trace:           False
    Precision
      result:        ap_fixed<18,6>
      weight:        ap_fixed<14,4>
      bias:          ap_fixed<14,4>
  activation
    Trace:           False
    Precision
      result:        ap_fixed<12,1>
  layers_1
    Trac

### 4.4 Verify generated artifacts

Primary output location:
`HLS4ML/<run>/myproject_prj/solution1/impl/vhdl`


In [12]:
vhdl_dir = HLS_OUTPUT_ABS / "myproject_prj" / "solution1" / "impl" / "vhdl"
terminal_log = HLS_OUTPUT_ABS / "terminal_output.txt"

print("HLS output dir:", HLS_OUTPUT_ABS)
print("terminal_output.txt exists:", terminal_log.exists())
print("VHDL dir exists:", vhdl_dir.exists())

if vhdl_dir.exists():
    vhdl_files = sorted(vhdl_dir.glob("*.vhd"))
    print("VHDL file count:", len(vhdl_files))
    for pth in vhdl_files[:12]:
        print(" -", pth.name)

reports = sorted(HLS_OUTPUT_ABS.glob("**/*.rpt"))
print("Report count:", len(reports))
for pth in reports[:12]:
    print(" -", pth.relative_to(HLS_OUTPUT_ABS))


HLS output dir: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/HLS4ML/Dense-7IN-32H1-32H2-1OUT-0_20260305_182640
terminal_output.txt exists: True
VHDL dir exists: True
VHDL file count: 8
 - dense_latency_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0.vhd
 - dense_latency_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_1.vhd
 - dense_latency_ap_fixed_12_1_5_3_0_ap_fixed_18_6_5_3_0_config8_0_0.vhd
 - linear_ap_fixed_18_6_5_3_0_ap_fixed_12_2_5_3_0_linear_config9_s.vhd
 - myproject.vhd
 - tanh_ap_fixed_18_6_5_3_0_ap_fixed_12_1_5_3_0_tanh_config4_s.vhd
 - tanh_ap_fixed_18_6_5_3_0_ap_fixed_12_1_5_3_0_tanh_config4_s_tanh_table1.vhd
 - tanh_ap_fixed_18_6_5_3_0_ap_fixed_12_1_5_3_0_tanh_config7_s.vhd
Report count: 30
 - myproject_prj/solution1/.autopilot/db/dense_latency_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0_0.verbose.bind.rpt
 - myproject_prj/solution1/.autopilot/db/dense_latency_0_0_0_0_0_0_0_0_0_0_0_0

## Step 5: Testing Model on PC (Local Hardware)

This step validates your trained model using the real cartpole from a local machine.
You will set:
- active controller selection
- serial-port behavior (optional manual override)
- neural-imitator model config

Calibration values from this step are reused in Step 6.

Please also review the calibration guide in [`README.md` (Calibration)](README.md#calibration).


### 5.0 Local Execution Requirement

Run this step on your own machine with the cartpole connected over USB. Do not run it from the lab server.

If needed, copy the repository from a lab server:

```bash
scp -r asuad\yourasuID@129.219.30.13:~/project/cartpole/physical-cartpole ~/Desktop/physical-cartpole
```


### 5.1 Set Controller to `neural-imitator`

Target file: `Driver/globals.py`

This helper cell updates `CONTROLLER_NAME` to `neural-imitator`.

In [13]:
from demo_helpers import patch_controller_name

APPLY_CONTROLLER_PATCH = True

# Optionally sets Driver/globals.py CONTROLLER_NAME to neural-imitator.
patch_controller_name(
    repo=REPO,
    apply_controller_patch=APPLY_CONTROLLER_PATCH,
)


globals.py: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Driver/globals.py
Current CONTROLLER_NAME: neural-imitator
Patched CONTROLLER_NAME -> neural-imitator


### 5.2 Serial Port Configuration (Optional Manual Override)

Target file: `Driver/DriverFunctions/interface.py`

If auto-detection fails (common on some macOS setups), this helper can add/use a manual serial-port override.

1. Set `SERIAL_PORT_OVERRIDE` to your device path.

Tip: on macOS, list ports with `ls /dev/tty.*` while the board is connected and powered.


In [ ]:
from demo_helpers import patch_manual_serial_override

SERIAL_PORT_OVERRIDE = "/dev/tty.usbserial-210351B7BD461"  # Replace with your local port path
APPLY_SERIAL_PATCH = True

# Optionally injects and sets MANUAL_SERIAL_PORT in DriverFunctions/interface.py.
patch_manual_serial_override(
    repo=REPO,
    serial_port_override=SERIAL_PORT_OVERRIDE,
    apply_serial_patch=APPLY_SERIAL_PATCH,
)


### 5.3 Model Configuration for `neural-imitator`

Target file: `Driver/CartPoleSimulation/Control_Toolkit_ASF/config_controllers.yml`

This helper updates key fields under `neural-imitator`:
- `PATH_TO_MODELS`
- `net_name`
- `input_precision`
- `hls4ml`

In [ ]:
from demo_helpers import patch_model_config

APPLY_MODEL_CONFIG_PATCH = True

# Optionally updates neural-imitator model path/name/precision in config_controllers.yml.
patch_model_config(
    repo=REPO,
    net_name=globals().get("SELECTED_NET_NAME", "Dense-7IN-32H1-32H2-1OUT-0"),
    path_to_models="./CartPoleSimulation/SI_Toolkit_ASF/Experiments/Experiment-1/Models/",
    apply_model_config_patch=APPLY_MODEL_CONFIG_PATCH,
)


### 5.4 Run PC Control Software

From the repository root (`physical-cartpole/`):

```bash
export PYTHONPATH=$(pwd):$PYTHONPATH
python Driver/control.py
```

Useful keys in the running app:
- `h`: help
- `K`: calibrate track middle
- `k`: PC control on/off
- `u`: chip control on/off
- `D`: dance mode on/off

Calibration outputs from this step (motor power, track middle behavior, vertical angle) are required for Step 6 firmware/SoC setup.


In [ ]:
import os

os.environ["PYTHONPATH"] = f"{REPO}:{os.environ.get('PYTHONPATH', '')}"
print("PYTHONPATH updated for this kernel.")
print("Run locally from repo root:")
print("  export PYTHONPATH=$(pwd):$PYTHONPATH")
print("  python Driver/control.py")


### 5.5 Calibration Notes

During this step, verify:
1. Middle of track calibration (`K`) each power cycle.
2. Motor power is sufficient to avoid sticking near boundaries.
3. Vertical angle/potentiometer settings are correct.

`ANGLE_HANGING_POLOLU` and `MOTOR_CORRECTION` values obtained here are reused in Step 6.


## Step 6: Implementation (Vivado/Vitis)

1. Generate the NN model bitstream (Vivado)
2. Generate the Zynq SoC project and `BOOT.bin` (Vitis/XSCT)


### 6.1 Preparation

This section prepares firmware/build scripts before Step 6.2/6.3.

- Confirm required files exist and Vivado/XSCT are visible in PATH.

Note:
- Optional patch cells write source files in-place; set their `APPLY_*` flags to `False` first for dry-run/preview output.


In [14]:
from demo_helpers import ensure_notebook_repo_context, preflight_step6

# Ensures Step 1.1 repo context exists and repo imports resolve from this checkout.
REPO_PATH = ensure_notebook_repo_context(globals())

# Checks required files and validates Vivado/tool visibility.
STEP6_PREFLIGHT = preflight_step6(
    repo=REPO_PATH,
    expected_vivado_version="2020.1",
    vivado_bin=globals().get("vivado_bin"),
)

vivado_exe = STEP6_PREFLIGHT["vivado_exe"]
vivado_version_line = STEP6_PREFLIGHT["vivado_version_line"]


vivado: /tools/Xilinx/Vivado/2020.1/bin/vivado
vivado -version: Vivado v2020.1 (64-bit)
xsct: /mnt/raid5/fpga/cad/xilinx/Vitis/2023.2/bin/xsct
Vivado version target: 2020.1
Vitis version target: 2020.1


### 6.2 Symlink and Script-Permissions Prep

This prep cell updates helper scripts before the build stages.

- It assumes the "NeuralImitator on Zynq" symlink block in `Firmware/create_symlinks_cartpole.sh` stays disabled; no symlink-script edits are applied.
- It sets executable mode bits on `scripts/install_zybo_board.sh`, `generate_bitstream.tcl`, and `generate_vitis_project.tcl`.

Files modified:
- mode bits on `scripts/install_zybo_board.sh`, `generate_bitstream.tcl`, `generate_vitis_project.tcl`

- This step is safe to rerun.


In [15]:
from demo_helpers import ensure_notebook_repo_context, run_step62_symlink_and_permissions_prep

# Ensures Step 1.1 repo context exists and repo imports resolve from this checkout.
REPO_PATH = ensure_notebook_repo_context(globals())

# Prepares script permissions and symlink-script readiness.
run_step62_symlink_and_permissions_prep(repo=REPO_PATH)


Assuming NeuralImitator symlink block stays disabled; no edits applied.
Executable bit set: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/scripts/install_zybo_board.sh
Executable bit set: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/tcl/generate_bitstream.tcl
Executable bit set: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/tcl/generate_vitis_project.tcl


### 6.3 Optional Firmware Value Patches

The next two code cells are optional convenience patches that change firmware values to match your current model/calibration outputs from Step 5.

1. Calibration patch cell:
updates `Firmware/Src/CartPoleFirmware/parameters.c` with Step 5 values (`MOTOR_CORRECTION`, `ANGLE_HANGING_POLOLU`).

2. Normalization/header patch cell:
updates normalization vectors and neuron-count defines in `Firmware/Src/Zynq/neural_imitator.c/.h` (or the hyphenated filename variants).

- These patches write files in place.
- Set `APPLY_PARAMETER_PATCH`, `APPLY_NORMALIZATION_PATCH`, and `UPDATE_NETWORK_HEADER` to `False` first if you want a dry run style preview before writing.


In [ ]:
# Optional
from demo_helpers import ensure_notebook_repo_context, apply_step63_parameter_patch

# Ensures Step 1.1 repo context exists and repo imports resolve from this checkout.
REPO_PATH = ensure_notebook_repo_context(globals())

APPLY_PARAMETER_PATCH = True
NEW_MOTOR_CORRECTION = [0.6310468, 0.0472680, 0.0408973]
NEW_ANGLE_HANGING_POLOLU = 783.0

# Optionally patches Step 5 calibration constants into parameters.c.
apply_step63_parameter_patch(
    repo=REPO_PATH,
    apply_parameter_patch=APPLY_PARAMETER_PATCH,
    new_motor_correction=NEW_MOTOR_CORRECTION,
    new_angle_hanging_pololu=NEW_ANGLE_HANGING_POLOLU,
)


In [ ]:
# Optional
from pathlib import Path
from demo_helpers import ensure_notebook_repo_context, apply_step63_normalization_patch

# Ensures Step 1.1 repo context exists and repo imports resolve from this checkout.
REPO_PATH = ensure_notebook_repo_context(globals())

APPLY_NORMALIZATION_PATCH = True
UPDATE_NETWORK_HEADER = True

MODEL_DIR = None
if "HLS_MODELS_DIR" in globals() and "SELECTED_NET_NAME" in globals():
    MODEL_DIR = Path(HLS_MODELS_DIR) / str(SELECTED_NET_NAME)

# Optionally patches normalization vectors and header neuron counts.
apply_step63_normalization_patch(
    repo=REPO_PATH,
    apply_normalization_patch=APPLY_NORMALIZATION_PATCH,
    update_network_header=UPDATE_NETWORK_HEADER,
    model_dir=MODEL_DIR,
    net_name=globals().get("NET_NAME", "Dense-7IN-32H1-32H2-1OUT-0"),
)


### 6.4 Generating the FPGA Bitstream

Before running Step 6.4/6.5, confirm these:
- Tool versions: Vivado 2020.1 and Vitis/XSCT 2020.1 are the expected versions for this flow.
- Key environment setup: `vivado` and `xsct` must be discoverable in `PATH` (typically by sourcing your Xilinx setup script, e.g. `scripts/setup_xilinx.sh`).
- Licensing env (if your installation requires it): ensure your license variable is configured (`XILINXD_LICENSE_FILE` or `LM_LICENSE_FILE`).
- Repo location assumption: original TCL/scripts assume `~/physical-cartpole`; this notebook auto-generates temporary TCL copies with corrected paths when your repo is elsewhere.
- Execution context: run Step 1.1 and Step 6.1 first so `REPO`/`REPO` are set and preflight checks are done.

This cell executes the README Step 6.2 commands directly from Python:
1. `cd ~/physical-cartpole && ./scripts/install_zybo_board.sh`
2. `cd ~/physical-cartpole/FPGA/VivadoProjects && vivado -mode batch -source ~/physical-cartpole/tcl/generate_bitstream.tcl`
3. Retry with `MALLOC_CHECK_`, `MALLOC_ARENA_MAX`, and `LD_PRELOAD` if the first Vivado run fails.

Important: Running this cell will take ~50-60 minutes


In [16]:
import shutil
from pathlib import Path
from demo_helpers import ensure_notebook_repo_context, run_step64_bitstream

# Ensures Step 1.1 repo context exists and repo imports resolve from this checkout.
REPO_PATH = ensure_notebook_repo_context(globals())

if "IMPL_VIVADO_EXE" not in globals():
    if "vivado_exe" in globals():
        IMPL_VIVADO_EXE = Path(vivado_exe)
        print("Using Vivado from Step 6.1 variable vivado_exe:", IMPL_VIVADO_EXE)
    else:
        vivado_on_path = shutil.which("vivado")
        if vivado_on_path:
            IMPL_VIVADO_EXE = Path(vivado_on_path)
            print("Using Vivado from PATH:", IMPL_VIVADO_EXE)
        else:
            raise RuntimeError("Run Step 6.1 first to resolve Vivado/XSCT for the pinned tool version.")

# Runs Vivado bitstream generation with existing retry behavior.
STEP64_RESULT = run_step64_bitstream(
    repo=REPO_PATH,
    impl_vivado_exe=Path(IMPL_VIVADO_EXE),
    hls_output_abs=Path(HLS_OUTPUT_ABS) if "HLS_OUTPUT_ABS" in globals() else None,
)

xsa = STEP64_RESULT["xsa"]


Using Vivado from Step 6.1 variable vivado_exe: /tools/Xilinx/Vivado/2020.1/bin/vivado
Executable bit set: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/scripts/install_zybo_board.sh
Executable bit set: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/tcl/generate_bitstream.tcl
Executable bit set: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/tcl/generate_vitis_project.tcl
Patched HLS4ML paths in: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/.notebook_impl/CartpoleDriverZynq_new.notebook.tcl
Using notebook-local TCL copy: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/.notebook_impl/generate_bitstream.notebook.tcl
Removed stale Vivado project directory: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/FPGA/VivadoProjects/CartpoleDriverZynq
$ bash /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/phys

Cloning into '/home/jacobj/.Xilinx/tmp-xilinx-boardstore'...


Installing Zybo Z7-20 board files to /home/jacobj/.Xilinx/board_repos/xilinx-zybo/boards/Digilent/zybo-z7-20/A.0...
Zybo Z7-20 board (version 1.0) installed from XilinxBoardStore branch 2020.1.
Board path: /home/jacobj/.Xilinx/board_repos/xilinx-zybo/boards/Digilent/zybo-z7-20/A.0

To verify in Vivado 2020.1 Tcl console:
  get_board_parts -filter {NAME == "digilentinc.com:zybo-z7-20:1.0"}
$ /tools/Xilinx/Vivado/2020.1/bin/vivado -mode batch -source /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/.notebook_impl/generate_bitstream.notebook.tcl
cwd: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/FPGA/VivadoProjects

****** Vivado v2020.1 (64-bit)
  **** SW Build 2902540 on Wed May 27 19:54:35 MDT 2020
  **** IP Build 2902112 on Wed May 27 22:43:36 MDT 2020
    ** Copyright 1986-2020 Xilinx, Inc. All Rights Reserved.

Sourcing tcl script '/home/jacobj/.Xilinx/Vivado/Vivado_init.tcl'
source /mnt/raid5/asic/projects/NU/sandboxes/jaco

### 6.5 Generating the SoC Project and `BOOT.bin`

This cell executes the README Step 6.3 command from Python using:
`cd ~/physical-cartpole && xsct tcl/generate_vitis_project.tcl > vitis_output.log 2>&1`

Stability notes for Vitis 2020.1:
- If `DISPLAY` is unset and `xvfb-run` is available, this cell runs `xsct` under `xvfb-run -a` from the start.
- It watches for log idle hangs and known crash markers
- It refuses to start if an older Step 6.5 `xsct` process is still running on the same TCL.


In [18]:
from pathlib import Path
from demo_helpers import ensure_notebook_repo_context, run_step65_bootbin

# Ensures Step 1.1 repo context exists and repo imports resolve from this checkout.
REPO_PATH = ensure_notebook_repo_context(globals())

# Runs XSCT flow and returns generated BOOT.bin paths.
STEP65_RESULT = run_step65_bootbin(
    repo=REPO_PATH,
    vitis_tcl_src=REPO_PATH / "tcl" / "generate_vitis_project.tcl",
    hls_output_abs=Path(HLS_OUTPUT_ABS) if "HLS_OUTPUT_ABS" in globals() else None,
)

boot_bins = STEP65_RESULT["boot_bins"]


Resolved Step 6.5 hw_xsa path: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/FPGA/VivadoProjects/CartpoleDriverZynq/cartpole_driver_design_wrapper.xsa exists= True
Resolved Step 6.5 raw_bit_path: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/FPGA/VivadoProjects/CartpoleDriverZynq/CartpoleDriverZynq.runs/impl_1/cartpole_driver_design_wrapper.bit exists= True
Using notebook-local TCL copy: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/.notebook_impl/generate_vitis_project.notebook.tcl
Step 6.5 hw_xsa path: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/FPGA/VivadoProjects/CartpoleDriverZynq/cartpole_driver_design_wrapper.xsa
Step 6.5 bitstream path: /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/FPGA/VivadoProjects/CartpoleDriverZynq/CartpoleDriverZynq.runs/impl_1/cartpole_driver_design_wrapper.bit
No DISPLAY detected; running Step 6.5 u

Notes:
- The cells above run the existing automation scripts directly via Python `subprocess`.
- If this repo is not located at `~/physical-cartpole`, the notebook automatically creates temporary TCL copies with corrected absolute paths.
- If Vivado/XSCT are not found, source your Xilinx setup script first (for example `scripts/setup_xilinx.sh`) and rerun

## Step 7
Load Image on SD card and onto FPGA

This is the final step.

Once the image is successfully loaded onto the SD card and FPGA, the system will be fully configured and ready for operation. There are four switches on the board: the two in the middle serve important functions. One of the switches calibrates the center of the track, while the other allows the cartpole to stabilize either in the upward or downward position. Play with them to see what happens!

Congratulations on completing the setup!


In [19]:
boot_bins = sorted((REPO / "Firmware" / "VitisProjects").glob("**/BOOT.bin"))
print("Found BOOT.bin files:", len(boot_bins))
for p in boot_bins:
    print(" -", p)

if not boot_bins:
    print("No BOOT.bin found. Re-run Step 6.3 first.")
else:
    print("Next (manual/local):")
    print("1) Copy BOOT.bin to SD card root partition")
    print("2) Set board boot mode to SD")
    print("3) Power-cycle board and verify behavior with switches")


Found BOOT.bin files: 1
 - /mnt/raid5/asic/projects/NU/sandboxes/jacobj/cartpole_demo/physical-cartpole/Firmware/VitisProjects/BOOT.bin
Next (manual/local):
1) Copy BOOT.bin to SD card root partition
2) Set board boot mode to SD
3) Power-cycle board and verify behavior with switches
